# Apple vs Samsung — Competitive Financial & Market Analysis

## Data Collection

**Data Sources:**
1. **Yahoo Finance (via `yfinance`)** — historical stock prices, volume, and financial statements for Apple (AAPL) and Samsung (005930.KS)
2. **FRED (Federal Reserve Economic Data, via `pandas-datareader`)** — macroeconomic context: US Consumer Price Index (CPI) and US 10-Year Treasury Rate

**Coverage:** 2015-01-01 to present

**Variable Definitions:**
- `Open`, `High`, `Low`, `Close`, `Volume` — daily OHLCV stock price data
- `Total Revenue`, `Net Income`, `Gross Profit`, `Operating Income` — annual income statement items
- `Total Assets`, `Total Liab`, `Total Stockholder Equity` — annual balance sheet items
- `Free Cash Flow`, `Operating Cash Flow` — annual cash flow items
- `CPIAUCSL` — US CPI, all urban consumers (monthly, seasonally adjusted)
- `DGS10` — US 10-Year Treasury Constant Maturity Rate (daily)

In [ ]:
import subprocess, sys

for pkg in ['yfinance', 'pandas', 'numpy', 'matplotlib', 'seaborn', 'pandas-datareader']:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

print('All packages ready.')

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import os
from datetime import datetime

START_DATE = '2015-01-01'
END_DATE   = datetime.today().strftime('%Y-%m-%d')

TICKERS = {
    'Apple':   'AAPL',
    'Samsung': '005930.KS'
}

print(f'Date range: {START_DATE} → {END_DATE}')

## Source 1 — Yahoo Finance: Historical Stock Prices

In [ ]:
price_data = {}

for company, ticker in TICKERS.items():
    print(f'Fetching {company} ({ticker})...')
    df = yf.download(ticker, start=START_DATE, end=END_DATE, auto_adjust=True, progress=False)
    if df.empty:
        print(f'  WARNING: No data returned for {ticker}')
    else:
        # Flatten multi-level columns if present
        df.columns = [col[0] if isinstance(col, tuple) else col for col in df.columns]
        df.index = pd.to_datetime(df.index)
        price_data[company] = df
        print(f'  {len(df)} rows | {df.index.min().date()} to {df.index.max().date()}')

prices_combined = pd.concat(price_data, axis=1)
prices_combined.to_csv('stock_prices.csv')
print('\nSaved: stock_prices.csv')
prices_combined.tail(3)

## Source 1 — Yahoo Finance: Financial Statements (Annual)

In [ ]:
financials = {}

for company, ticker in TICKERS.items():
    print(f'Fetching financials for {company} ({ticker})...')
    t = yf.Ticker(ticker)

    income_stmt   = t.financials
    balance_sheet = t.balance_sheet
    cash_flow     = t.cashflow

    financials[company] = {
        'income_stmt':   income_stmt,
        'balance_sheet': balance_sheet,
        'cash_flow':     cash_flow
    }

    for label, df in [('income_stmt', income_stmt), ('balance_sheet', balance_sheet), ('cash_flow', cash_flow)]:
        if df is not None and not df.empty:
            fname = f'{company.lower()}_{label}.csv'
            df.to_csv(fname)
            print(f'  Saved {fname}  shape={df.shape}')
        else:
            print(f'  WARNING: {company} {label} is empty')

## Source 2 — FRED: Macroeconomic Indicators

**Link:** https://fred.stlouisfed.org/

Series fetched:
- **CPIAUCSL** — Consumer Price Index for All Urban Consumers (monthly, seasonally adjusted)
- **DGS10** — 10-Year Treasury Constant Maturity Rate (daily)

In [ ]:
import pandas_datareader.data as web

fred_series = {
    'CPI_US':       'CPIAUCSL',
    'Treasury_10Y': 'DGS10'
}

macro_data = {}

for name, series_id in fred_series.items():
    print(f'Fetching FRED series {series_id} ({name})...')
    try:
        df = web.DataReader(series_id, 'fred', START_DATE, END_DATE)
        df.index = pd.to_datetime(df.index)
        df.columns = [name]
        macro_data[name] = df
        fname = f'fred_{name.lower()}.csv'
        df.to_csv(fname)
        print(f'  {len(df)} rows | {df.index.min().date()} to {df.index.max().date()} | Saved {fname}')
    except Exception as e:
        print(f'  ERROR fetching {series_id}: {e}')

if macro_data:
    macro_combined = pd.concat(macro_data.values(), axis=1)
    macro_combined.to_csv('fred_macro.csv')
    print('\nSaved: fred_macro.csv')
    display(macro_combined.tail(5))

## Data Summary

In [ ]:
print('=== Downloaded CSV Files ===')
for f in sorted(f for f in os.listdir('.') if f.endswith('.csv')):
    rows = len(pd.read_csv(f))
    print(f'  {f:<48}  {rows:>6} rows')

print('\n=== Apple Stock (head) ===')
if 'Apple' in price_data:
    display(price_data['Apple'].head(3))

print('\n=== Samsung Stock (head) ===')
if 'Samsung' in price_data:
    display(price_data['Samsung'].head(3))

print('\n=== Macro Data (head) ===')
if macro_data:
    display(macro_combined.head(3))